# M3L4 E10 — LangGraph multiagente supervisor + Langfuse [OK] Resolution
### Módulo 3 · Lecture 4 · Construcción, pruebas y trazabilidad de agentes en producción

## ¿Por qué importa este ejercicio?

En E09 el router elegía un agente y terminaba. Pero en sistemas reales un agente puede necesitar **derivar a otro agente** (handoff), y eso puede generar **loops** si no se controla.

Acá introducimos un **supervisor** que:
1. Clasifica la consulta
2. Mantiene un registro de qué agentes ya visitó
3. Detecta si ya se visitó un agente (para evitar loops)
4. Decide si continuar o terminar

| Concepto | Definición simple | Cómo aparece acá |
|---|---|---|
| **Supervisor** | Nodo que orquesta y decide el flujo | `supervisor_node()` asigna intent y decide si terminar |
| **visited_agents** | Lista de agentes ya ejecutados | `state['visited_agents']` se actualiza en cada nodo |
| **Handoff** | Derivación de un agente a otro | `supervisor_route()` decide a qué agente ir |
| **Loop prevention** | Detectar que un agente ya se visitó | `target in visited` -> `done = True` |

In [ ]:
!pip install -q langfuse langchain langchain-openai langgraph
print('Instalación completa.')

In [ ]:
import os
from getpass import getpass
os.environ['LANGFUSE_PUBLIC_KEY'] = getpass('Langfuse Public Key: ')
os.environ['LANGFUSE_SECRET_KEY'] = getpass('Langfuse Secret Key: ')
os.environ['LANGFUSE_BASE_URL']   = 'https://cloud.langfuse.com'
os.environ['OPENAI_API_KEY']      = getpass('OpenAI API Key: ')
print('OK.')

In [ ]:
from typing import List
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, END
from langfuse.langchain import CallbackHandler

def route_query_v2(query):
    q = query.lower()
    det = []
    if any(w in q for w in ['vacaciones','licencia','recibo','nómina','rrhh']): det.append('hr')
    if any(w in q for w in ['vpn','error','app','laptop','wifi','login','contraseña']): det.append('it')
    if any(w in q for w in ['factura','pago','reembolso','gasto','cobro','comprobante','salario']): det.append('finance')
    if any(w in q for w in ['contrato','legal','confidencialidad','nda','acuerdo']): det.append('legal')
    if len(det) > 1: return 'multi_intent'
    if len(det) == 1: return det[0]
    if len(q.split()) <= 2: return 'clarification'
    return 'general'

class SupervisorState(TypedDict):
    query: str
    intent: str
    visited_agents: List[str]
    response: str
    done: bool

print('Setup listo.')

In [ ]:
def supervisor_node(state):
    intent = route_query_v2(state['query'])
    visited = state.get('visited_agents', []).copy()
    intent_to_agent = {'hr':'HRAgent','it':'ITAgent','finance':'FinanceAgent','legal':'LegalAgent'}
    target = intent_to_agent.get(intent)
    done = (target is None) or (target in visited)
    return {'intent': intent, 'visited_agents': visited, 'done': done}

def hr_agent_node(state):
    visited = state.get('visited_agents', []).copy()
    visited.append('HRAgent')
    return {'response': 'HRAgent: Para vacaciones o recibos, ingresá al portal de RRHH.', 'visited_agents': visited, 'done': True}

def it_agent_node(state):
    visited = state.get('visited_agents', []).copy()
    visited.append('ITAgent')
    return {'response': 'ITAgent: Para VPN o errores técnicos, revisá conexión y abrí ticket.', 'visited_agents': visited, 'done': True}

def finance_agent_node(state):
    visited = state.get('visited_agents', []).copy()
    visited.append('FinanceAgent')
    return {'response': 'FinanceAgent: Para facturas o pagos, revisá el portal de facturación.', 'visited_agents': visited, 'done': True}

def legal_agent_node(state):
    visited = state.get('visited_agents', []).copy()
    visited.append('LegalAgent')
    return {'response': 'LegalAgent: Para contratos o NDAs, contactá al equipo legal.', 'visited_agents': visited, 'done': True}

def supervisor_route(state):
    if state.get('done'):
        return 'end'
    return {'hr':'hr_agent','it':'it_agent','finance':'finance_agent','legal':'legal_agent'}.get(state['intent'],'end')

print('Nodos listos.')

## Solución — Grafo con supervisor

El flujo es:

```
START -> supervisor_node (clasifica + decide si ya visitó) 
         -> conditional_edge
           -> hr_agent / it_agent / finance_agent / legal_agent (ejecutan y marcan done=True)
           -> END (si ya estaba visitado o es un intent no manejado)
```

La clave está en `visited_agents`: cada agente se agrega a la lista al ejecutarse. El supervisor chequea esa lista antes de derivar.

In [ ]:
builder = StateGraph(SupervisorState)
builder.add_node('supervisor_node',    supervisor_node)
builder.add_node('hr_agent',           hr_agent_node)
builder.add_node('it_agent',           it_agent_node)
builder.add_node('finance_agent',      finance_agent_node)
builder.add_node('legal_agent',        legal_agent_node)
builder.set_entry_point('supervisor_node')
builder.add_conditional_edges(
    'supervisor_node',
    supervisor_route,
    {'hr_agent':'hr_agent','it_agent':'it_agent','finance_agent':'finance_agent','legal_agent':'legal_agent','end': END}
)
for node in ['hr_agent','it_agent','finance_agent','legal_agent']:
    builder.add_edge(node, END)
graph = builder.compile()
print('Grafo compilado.')
print(graph.get_graph().draw_mermaid())

## Ejecución con Langfuse

Cada consulta muestra:
- El intent detectado por el supervisor
- Qué agente se ejecutó (o si ya fue visitado)
- La respuesta

In [ ]:
for q in ['¿Cómo solicito mis días de vacaciones?','Mi VPN no conecta','Necesito ver mi factura','ayuda']:
    lf = CallbackHandler()
    out = graph.invoke(
        {'query': q, 'intent': '', 'visited_agents': [], 'response': '', 'done': False},
        config={'callbacks': [lf], 'metadata': {'langfuse_tags': ['m3l4','supervisor']}}
    )
    print(f'Query: {q[:40]}  |  Intent: {out["intent"]}  |  Visited: {out["visited_agents"]}')
    print(f'  -> {out["response"][:70]}...')
    print()

## Verificación

In [ ]:
lf = CallbackHandler()
r = graph.invoke({'query': 'No puedo ver mi factura','intent':'','visited_agents':[],'response':'','done':False},
                 config={'callbacks': [lf]})
assert r['intent'] == 'finance'
assert 'FinanceAgent' in r['visited_agents']
assert r['done'] == True
assert len(r['response']) > 5
print('Checks E10 OK')

## [OK] Cierre — ¿Qué logramos?

| Mecanismo | Problema que resuelve |
|---|---|
| **visited_agents** | Evita que el mismo agente se ejecute dos veces (loops) |
| **Supervisor** | Centraliza la lógica de routing y control de flujo |
| **Conditional edge a END** | Si el intent no es manejable, termina sin ejecutar nada |

> **Diferencia con E09:** acá el supervisor es un nodo separado que decide el flujo. En E09 el router y la decisión estaban en el mismo `route_to_node`.

**¿Qué sigue?** En E11 vamos a integrar el **golden dataset** con Langfuse Scores para medir accuracy automáticamente.